# CoCo's Library — Notifications API Walkthrough (#6)

A frontend-facing walkthrough of the in-app **notification** feature: the
notification inbox and the three community notification types — upload
confirmation, like milestone, and new CoCo's Original.

> The API returns only `type` + `payload`; the **frontend decides navigation**
> from those (the backend builds no deep-links).

**Scope:** feeds / community notifications only. Trial-ending (subscription),
inactive-welcome (account), and the verification / password / email-change
emails are out of scope here.

> Secrets in the setup cell are placeholders — fill them in for your environment.

## 0 · Setup

In [1]:
import json
import httpx

BASE_URL = "<FEEDS-HOST>"            # feeds host (no trailing slash)
API = f"{BASE_URL}/api/v1/feeds"

TEST_EMAIL = "<TEST-EMAIL>"
TEST_PASSWORD = "<REDACTED>"
FIREBASE_API_KEY = "<FIREBASE-WEB-API-KEY>"                   # public Firebase Web API key

DEMO_STORY_ID = "aaaa1111-1111-1111-1111-111111111111"  # a published story at 9 likes
DEMO_TASK_ID  = "bbbb2222-2222-2222-2222-222222222222"  # a completed generation task

http = httpx.Client(timeout=20)

def show(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False))


### Sign in to Firebase
The feeds API authenticates via a Firebase ID token (same as the app).

In [2]:
r = http.post(
    f"https://identitytoolkit.googleapis.com/v1/accounts:signInWithPassword?key={FIREBASE_API_KEY}",
    json={"email": TEST_EMAIL, "password": TEST_PASSWORD, "returnSecureToken": True},
)
TOKEN = r.json()["idToken"]
H = {"Authorization": f"Bearer {TOKEN}"}
print("signed in")

signed in


## 1 · The inbox  ·  `GET /notifications`

Newest-first notifications for the signed-in (parent) account, with an
`unread_count`. Cursor-paginated via `next_cursor` / `has_more`.

In [3]:
show(http.get(f"{API}/notifications", headers=H).json())

{
  "items": [],
  "unread_count": 0,
  "next_cursor": null,
  "has_more": false
}


## 2 · New CoCo's Original broadcast (B5)  ·  admin only

`POST /admin/notify-new-originals` fans out one in-app notification to every
user. It is **manual** (the CoCo team triggers it after publishing Originals)
and **rate-limited to once per ISO week** — a second call the same week
notifies nobody.

In [4]:
show(http.post(f"{API}/admin/notify-new-originals", headers=H).json())   # first call this week
show(http.post(f"{API}/admin/notify-new-originals", headers=H).json())   # same week -> notified: 0

{
  "notified": 7
}
{
  "notified": 0
}


In [5]:
show(http.get(f"{API}/notifications", headers=H).json())   # now contains a new_original

{
  "items": [
    {
      "id": "149bdd4d-4f13-4c75-a8fb-174857be395a",
      "type": "new_original",
      "title": "A brand new CoCo's original story is here!",
      "body": "Come explore in CoCo's Library!",
      "payload": {
        "section": "originals"
      },
      "is_read": false,
      "created_at": "2026-05-30T09:17:35.565807Z"
    }
  ],
  "unread_count": 1,
  "next_cursor": null,
  "has_more": false
}


## 3 · Like milestone (B4)

When a like takes a story across **10 / 50 / 100 / 500 / 1K / 5K**, the story
**owner** is notified. Only the highest milestone crossed fires, and each
(story, milestone) fires at most once. `DEMO_STORY_ID` sits at 9 likes, so one
like crosses 10.

In [6]:
show(http.post(f"{API}/stories/{DEMO_STORY_ID}/like", headers=H).json())   # like_count -> 10
show(http.get(f"{API}/notifications", headers=H).json())                   # like_milestone (10)

{
  "liked": true,
  "like_count": 10
}
{
  "items": [
    {
      "id": "18d90bdd-6dff-4134-8c29-fbcfb477de2b",
      "type": "like_milestone",
      "title": "Staging Almost Famous just got 10 likes! 🎉",
      "body": "",
      "payload": {
        "story_id": "aaaa1111-1111-1111-1111-111111111111",
        "milestone": 10
      },
      "is_read": false,
      "created_at": "2026-05-30T09:17:36.187642Z"
    },
    {
      "id": "149bdd4d-4f13-4c75-a8fb-174857be395a",
      "type": "new_original",
      "title": "A brand new CoCo's original story is here!",
      "body": "Come explore in CoCo's Library!",
      "payload": {
        "section": "originals"
      },
      "is_read": false,
      "created_at": "2026-05-30T09:17:35.565807Z"
    }
  ],
  "unread_count": 2,
  "next_cursor": null,
  "has_more": false
}


## 4 · Upload confirmation (B3)

Publishing a story notifies the parent in-app. (The backend also best-effort
**emails** the parent a one-tap link to undo an accidental share — that link is
generated and signed server-side and delivered by email; it is not a
frontend-called API, so it is out of scope for this walkthrough.)

In [7]:
pub = http.post(f"{API}/stories", headers=H, json={
    "task_id": DEMO_TASK_ID, "title": "My Bedtime Adventure", "category": "adventure_fantasy",
}).json()
show(pub)
show(http.get(f"{API}/notifications", headers=H).json())   # upload_confirm

{
  "id": "6b8d7676-9a50-4460-a7a0-317b9e94c6ff",
  "title": "My Bedtime Adventure",
  "cover_image_url": "https://s/1.jpg",
  "status": "published",
  "created_at": "2026-05-30T09:17:36.588851Z"
}


{
  "items": [
    {
      "id": "329ecce6-8de9-49a3-b0be-d74701ae8c78",
      "type": "upload_confirm",
      "title": "My Bedtime Adventure has been shared to CoCo's Library!",
      "body": "Other families can now read it.",
      "payload": {
        "story_id": "6b8d7676-9a50-4460-a7a0-317b9e94c6ff"
      },
      "is_read": false,
      "created_at": "2026-05-30T09:17:36.594002Z"
    },
    {
      "id": "18d90bdd-6dff-4134-8c29-fbcfb477de2b",
      "type": "like_milestone",
      "title": "Staging Almost Famous just got 10 likes! 🎉",
      "body": "",
      "payload": {
        "story_id": "aaaa1111-1111-1111-1111-111111111111",
        "milestone": 10
      },
      "is_read": false,
      "created_at": "2026-05-30T09:17:36.187642Z"
    },
    {
      "id": "149bdd4d-4f13-4c75-a8fb-174857be395a",
      "type": "new_original",
      "title": "A brand new CoCo's original story is here!",
      "body": "Come explore in CoCo's Library!",
      "payload": {
        "section": "origi

## 5 · Inbox read flow
`POST /notifications/{id}/read` and `POST /notifications/read-all`.

In [8]:
nid = http.get(f"{API}/notifications", headers=H).json()["items"][0]["id"]
show(http.post(f"{API}/notifications/{nid}/read", headers=H).json())   # {"marked": 1}
show(http.post(f"{API}/notifications/read-all", headers=H).json())     # {"marked": N}
show(http.get(f"{API}/notifications", headers=H).json())               # unread_count: 0

{
  "marked": 1
}
{
  "marked": 2
}


{
  "items": [
    {
      "id": "329ecce6-8de9-49a3-b0be-d74701ae8c78",
      "type": "upload_confirm",
      "title": "My Bedtime Adventure has been shared to CoCo's Library!",
      "body": "Other families can now read it.",
      "payload": {
        "story_id": "6b8d7676-9a50-4460-a7a0-317b9e94c6ff"
      },
      "is_read": true,
      "created_at": "2026-05-30T09:17:36.594002Z"
    },
    {
      "id": "18d90bdd-6dff-4134-8c29-fbcfb477de2b",
      "type": "like_milestone",
      "title": "Staging Almost Famous just got 10 likes! 🎉",
      "body": "",
      "payload": {
        "story_id": "aaaa1111-1111-1111-1111-111111111111",
        "milestone": 10
      },
      "is_read": true,
      "created_at": "2026-05-30T09:17:36.187642Z"
    },
    {
      "id": "149bdd4d-4f13-4c75-a8fb-174857be395a",
      "type": "new_original",
      "title": "A brand new CoCo's original story is here!",
      "body": "Come explore in CoCo's Library!",
      "payload": {
        "section": "origina

## Notification reference

| `type` | fired when | recipient | `payload` |
|---|---|---|---|
| `upload_confirm` | a story is published | the publisher (parent) | `{ "story_id" }` |
| `like_milestone` | likes cross 10 / 50 / 100 / 500 / 1K / 5K | the story owner | `{ "story_id", "milestone" }` |
| `new_original` | admin broadcasts new Originals (weekly max) | every user | `{ "section": "originals" }` |

The frontend maps `type` + `payload` to a destination (open the story, or the Originals tab). The backend never returns a deep-link.